<a href="https://colab.research.google.com/github/herlocksholmes1888/llm-text-detection/blob/main/LLM_Detection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LLM Text Detector** 🤖
This is a Machine Learning project, created with Kaggle's dataset, made to detect when it's analyzing text that was generated by Artificial Intelligence or when it was created by a Human.

There are a couple of blindspots to be considered, such as the following:

1️⃣ GenAI learned to write texts with human input. There are some human texts that may get a false positive for that reason, especially if the writing has "AI quirks" (extensive usage of —, for example).
2️⃣

## **Project setup** 💻

These steps are necessary to fetch the necessary data for the experiment. Here, we are connected to a Drive with an extensive dataset of human and AI generated texts provided by Kaggle.

In [ ]:
# Google Drive connection
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Project imports
import pandas as pd

In [ ]:
# Training data
raw = '/content/drive/MyDrive/Human Text vs LLM Text Detection/train_v2_drcat_02.csv'
df = pd.read_csv(raw)

df

,text,label,prompt_name,source,RDizzl3_seven
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False
...,...,...,...,...,...
44863,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44864,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44865,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44866,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True


## **Exploratory analysis**

In [ ]:
# Is there a correlation between `label` and `RDizzl3_seven`?
label_rd_corr = df['label'].corr(df['RDizzl3_seven'])

# There doesn't seem to be any correlation between the two columns
# RDizzl3 appears to be an irrelevant column in the analysis we intend to do
label_rd_corr

np.float64(-0.16283924007362083)

In [ ]:
# Are there any null values?
missing_values = df.isnull().sum()

# There aren't any null values in this dataset
missing_values


,0
text,0
label,0
prompt_name,0
source,0
RDizzl3_seven,0


In [ ]:
# Isolate relevant columns in df
df = df[['text', 'label']]

# For this analysis, only two columns will be necessary: text, which has the content to be analysed, and label, which says
# whether or not the text has been written by a person
df

,text,label
0,Phones\n\nModern humans today are always on th...,0
1,This essay will explain if drivers should or s...,0
2,Driving while the use of cellular devices\n\nT...,0
3,Phones & Driving\n\nDrivers should not be able...,0
4,Cell Phone Operation While Driving\n\nThe abil...,0
...,...,...
44863,"Dear Senator,\n\nI am writing to you today to ...",1
44864,"Dear Senator,\n\nI am writing to you today to ...",1
44865,"Dear Senator,\n\nI am writing to you today to ...",1
44866,"Dear Senator,\n\nI am writing to you today to ...",1


In [ ]:
# What's the size difference between samples?
human_texts = df.loc[df['label'] == 0, "text"]
llm_texts = df.loc[df['label'] == 1, "text"]

human_texts_length = human_texts.str.len()
llm_texts_length = llm_texts.str.len()

# How long is the longest text written by a human? How small is the smallest text written by a human?
# What is the average of characters written by a human?
largest_human_text = human_texts_length.max()
average_human_text = human_texts_length.mean()
smallest_human_text = human_texts_length.min()

print(f'Largest human text: {largest_human_text}')
print(f'Smallest human text: {smallest_human_text}')
print(f'Average human text: {average_human_text}')
print('-----------')

# How long is the longest text written by an LLM? How small is the smallest text written by an LLM?
# What is the average of characters written by a LLM?
largest_llm_text = llm_texts_length.max()
average_llm_text = llm_texts_length.mean()
smallest_llm_text = llm_texts_length.min()

print(f'Largest LLM text: {largest_llm_text}')
print(f'Smallest LLM text: {smallest_llm_text}')
print(f'Average LLM text: {average_llm_text}')

# Q: Is there any significant difference between the length of a text written by a human and the length of a text written by an LLM?
# A: Yes. Humans tend to write more words than an LLM.
#
# Are the added words redundant?
# Hypothesis: LLM text tends to be more succint, whilst humans tend to embelish what they write.
# This can be a useful pattern to teach a ML model.

Largest human text: 18322
Smallest human text: 691
Average human text: 2348.503890979504
-----------
Largest LLM text: 5078
Smallest LLM text: 48
Average LLM text: 2009.2924501343086


## **Machine Learning**